In [1]:
import sys
import os
sys.path.append('../')  # Ensure Python can find the 'Scripts' folder
import numpy as np
from Scripts.visualization import *

pairwise = np.load("../results/metrics_Pairwise_noise.npz")
pairwise_diffusive = np.load("../results/metrics_Pairwise_diffusive_noise.npz")
couplin = np.load("../results/metrics_Coupling_noise.npz")
couplin_diffusive = np.load("../results/metrics_Coupling_diffusive_noise.npz")

metrics = ["oscillation_map","TC","DTC","cumulant","powercorr","skew","kurt"]

In [2]:
import plotly.graph_objects as go
import numpy as np

def plot_metrics_contour(matrix, P_values, K_values, title="Metric Map",
                         colorscale="Inferno", n_contours=15, horizontal_lines=False,P1=0,P2=0):

    z_min = np.nanmin(matrix)
    z_max = np.nanmax(matrix)

    fig = go.Figure(data=go.Contour(
        z=matrix,
        x=K_values,
        y=P_values,
        colorscale=colorscale,
        contours=dict(
            start=z_min,
            end=z_max,
            size=(z_max - z_min) / n_contours,
            coloring='heatmap',
            showlines=True
        ),
        colorbar=dict(
            thickness=12,
            len=0.8
        )
    ))

    if horizontal_lines:
        # líneas horizontales
        fig.add_hline(y=P1, line=dict(color="lightblue", width=3), line_dash="dash")
        fig.add_hline(y=P2, line=dict(color="lightblue", width=3), line_dash="dash")
    
    fig.update_layout(
        title=title,
        xaxis_title='K',
        yaxis_title='P',
        width=500,
        height=400,
        margin=dict(l=60, r=40, t=50, b=50)
    )

    fig.show()
    fig.write_image(f"imgs/{title}.png")

In [3]:
for m in metrics:
    plot_metrics_contour(pairwise[m] , pairwise["P"], pairwise["K3"], title=f"{m} Pairwise")
    plot_metrics_contour(couplin[m] , pairwise["P"], pairwise["K3"], title=f"{m} coupling")
    plot_metrics_contour(pairwise_diffusive[m] , pairwise["P"], pairwise["K3"], title=f"{m} Pairwise Diffusive")
    plot_metrics_contour(couplin_diffusive[m] , pairwise["P"], pairwise["K3"], title=f"{m} coupling Diffusive")

In [30]:
metrics = ["oscillation_map","TC","DTC","cumulant","powercorr","skew","kurt"]
def normalize_z(M):
    return (M - np.nanmean(M)) / np.nanstd(M)


#def normalize_minmax(M):
#    return (M - np.nanmin(M)) / (np.nanmax(M) - np.nanmin(M))


diff = {}
diff_diffusive = {}

for m in metrics:
    
    if m == 'oscillation_map':
        pair_new = np.where(pairwise[m] == 0, -1, pairwise[m])
        coupling_new = np.where(couplin[m] == 0, -2, couplin[m])

        pair_diffusive_new = np.where(pairwise_diffusive[m] == 0, -1, pairwise_diffusive[m])
        coupling_diffusive_new = np.where(couplin_diffusive[m] == 0, -2, couplin_diffusive[m])
        
        diff[m] = coupling_new - pair_new
        diff_diffusive[m] = coupling_diffusive_new - pair_diffusive_new
        plot_metrics_contour(diff[m] , pairwise["P"], pairwise["K3"], title=f"{m} difference",horizontal_lines=True,P1=6,P2=9)
        plot_metrics_contour(diff_diffusive[m] , pairwise["P"], pairwise["K3"], title=f"{m} diffusive difference",horizontal_lines=True,P1=6.5,P2=9)

    else:
        pair = normalize_z(pairwise[m])
        coup = normalize_z(couplin[m])

        pair_d = normalize_z(pairwise_diffusive[m])
        coup_d = normalize_z(couplin_diffusive[m])

        diff[m] = coup - pair
        diff_diffusive[m] = coup_d - pair_d


        plot_metrics_contour(diff[m] , pairwise["P"], pairwise["K3"], title=f"{m} difference",horizontal_lines=True,P1=6,P2=9)
        plot_metrics_contour(diff_diffusive[m] , pairwise["P"], pairwise["K3"], title=f"{m} diffusive difference",horizontal_lines=True,P1=6.5,P2=9)

In [22]:
metrics = ["TC","DTC","cumulant","powercorr","skew","kurt"]
rms_values = {}

# calcular RMS de cada métrica
for m in metrics:
    rms_values[m] = np.sqrt(np.mean(diff[m]**2))

# ordenar de mayor a menor
rms_sorted = sorted(rms_values.items(), key=lambda x: x[1], reverse=True)

# imprimir ordenado
for m, v in rms_sorted:
    print(f"{m}: {v:.4f}")

powercorr: 0.8849
TC: 0.6237
cumulant: 0.5672
DTC: 0.5380
kurt: 0.4667
skew: 0.4042


In [23]:
metrics = ["TC","DTC","cumulant","powercorr","skew","kurt"]
rms_values = {}

# calcular RMS de cada métrica
for m in metrics:
    rms_values[m] = np.sqrt(np.mean(diff_diffusive[m]**2))

# ordenar de mayor a menor
rms_sorted = sorted(rms_values.items(), key=lambda x: x[1], reverse=True)

# imprimir ordenado
for m, v in rms_sorted:
    print(f"{m}: {v:.4f}")

cumulant: 1.8428
DTC: 0.8096
powercorr: 0.7770
TC: 0.7769
kurt: 0.4673
skew: 0.3080


In [35]:
!pip install nbconvert